In [ ]:
# Meta Prompt

# 1. Annotate 30 wrong samples you will use for evaluation.
# 2. Evaluate the samples to get the initial results using LLM.
# 3. Instruct a reasoning model to optimize the prompts, presenting the wrong samples and result.
# 4. Feed some/all of the samples into your system with improved prompts to generate new outputs.
# 5. Use a reasoning model to evaluate the results.
# 6. Check to see if the results improves.
# 7. Repeat until desired result.

In [4]:
# import pandas as pd

# examples = pd.read_csv("meta_examples.csv")
# # Drop first two columns
# examples = examples.iloc[:, 2:]

# # Show first 3 rows
# examples.head(3)

In [12]:
PROMPT_TEMPLATE = '''
In this task, you will evaluate the quality of the Text in relation to the given Triple Set. How well does the Text represent the Triple Set?  You will be given four specific Dimensions to evaluate against:

Dimensions:"""
No-Omissions: ALL the information in the Triple Set is present in the Text.
No-Additions: ONLY information from the Triple Set is present in the Text.
Grammaticality: The Text is free of grammatical and spelling errors.
Fluency: The Text flows well and is easy to read; its parts are connected in a natural way."""

Important note on No-Omissions and No-Additions: some Triple Set/Text pairs contain non-factual information and even fictional names for people, places, dates, etc. Whether there are omissions and/or additions in a Text is NOT related to factual truth, but instead is strictly related to the contents of the input Triple Set.
Important note on Grammaticality and Fluency: for Grammaticality and Fluency you do not need to consider the input Triple Set; only the intrinsic quality of the Text needs to be assessed.

You need to provide the scores ranging from 1 (indicating the lowest score) to 7 (indicating the highest score) for each of the dimensions and a short justification for each score in the following JSON format:  {{"No-Omissions": {{"Justification": "", "Score": ""}}, "No-Additions": {{"Justification": "", "Score": ""}}, "Grammaticality": {{"Justification": "", "Score": ""}}, "Fluency": {{"Justification": "", "Score": ""}} }}.

Make sure to read thoroughly the Triple Set and the {language} Text below, and assess the four Dimensions using the instructions and template above.

Triple Set: {Triples} \nText: {Nice_Text} \n\n
'''

In [ ]:
import os
import time
import json
import argparse
import pandas as pd

from typing import Dict, Any, Optional
from agents.llm_model import UnifiedModel, model_name

# === JSON extractor ===
def extract_first_json(s: str) -> Optional[Dict[str, Any]]:
    if not s or '{' not in s:
        return None
    start = s.find('{')
    stack = 0
    for i, ch in enumerate(s[start:], start=start):
        if ch == '{':
            stack += 1
        elif ch == '}':
            stack -= 1
            if stack == 0:
                candidate = s[start:i+1]
                try:
                    return json.loads(candidate)
                except json.JSONDecodeError:
                    try:
                        cleaned = candidate.replace(",}", "}").replace(",]", "]")
                        return json.loads(cleaned)
                    except Exception:
                        return None
    return None

# === Evaluator ===
def evaluate_row(unified: UnifiedModel, triples: str, text: str) -> Dict[str, Any]:
    prompt = PROMPT_TEMPLATE.format(Triples=triples, Nice_Text=text, language="English")
    chain = unified.model_("You are a careful evaluator. Return only JSON.")
    try:
        response = chain.invoke({"input": prompt})
        raw_text = response.content if hasattr(response, "content") else str(response)
    except Exception as e:
        print(f"LLM call failed: {e}")
        raw_text = ""
    parsed = extract_first_json(raw_text)
    return {"raw": raw_text, "parsed": parsed}

# === Main ===
def main(args):
    df = pd.read_csv(args.input)
    if args.drop_first_two:
        df = df.iloc[:, 2:]

    triples_col, text_col = args.triples_col, args.text_col

    # Init model
    provider = "openai"
    conf = model_name.get(provider.lower(), {}).copy()
    conf["model_name"], conf["temperature"] = "o3", 0.0
    unified = UnifiedModel(provider=provider, **conf)

    results = []
    for i, row in df.iterrows():
        triples = str(row[triples_col])
        text = str(row[text_col])

        result = evaluate_row(unified, triples, text)
        parsed = result["parsed"]

        res = {
            "index": i,
            triples_col: triples,
            text_col: text,
            "llm_raw": result["raw"],
        }

        if parsed:
            # Store parsed values explicitly
            for dim in ["No-Omissions", "No-Additions", "Grammaticality", "Fluency"]:
                if dim in parsed:
                    res[f"{dim}_score"] = parsed[dim].get("Score", "")
                    res[f"{dim}_just"] = parsed[dim].get("Justification", "")

        results.append(res)

        if (i+1) % 10 == 0:
            print(f"Processed {i+1}/{len(df)}")
        if args.limit and i+1 >= args.limit:
            break
        time.sleep(args.pause)

    # Save detailed row-level results
    results_df = pd.DataFrame(results)
    results_df.to_csv(args.output, index=False)
    print(f"Saved {len(results)} evaluations to {args.output}")

    # Compute averages
    numeric_cols = [c for c in results_df.columns if c.endswith("_score")]
    avg_scores = results_df[numeric_cols].apply(pd.to_numeric, errors="coerce").mean()
    avg_df = avg_scores.reset_index()
    avg_df.columns = ["Dimension", "Average_Score"]
    avg_df.to_csv("averages.csv", index=False)
    print("=== Average Scores ===")
    print(avg_df)

# if __name__ == "__main__":
#     p = argparse.ArgumentParser()
#     p.add_argument("--input", default="meta_examples.csv")
#     p.add_argument("--output", default="evaluations.csv")
#     p.add_argument("--triples_col", default="Triples")
#     p.add_argument("--text_col", default="Output")
#     p.add_argument("--drop_first_two", action="store_true")
#     p.add_argument("--pause", type=float, default=0.5)
#     p.add_argument("--limit", type=int, default=0)
#     args = p.parse_args()
#     if args.limit == 0:
#         args.limit = None
#     main(args)

class Args:
    input = "meta_examples.csv"
    output = "evaluations.csv"
    triples_col = "Triples"
    text_col = "Output"
    drop_first_two = True
    pause = 0.5
    limit = None
    
args = Args()
main(args)

Processed 10/30
Processed 20/30
Processed 30/30
Saved 30 evaluations to evaluations.csv
=== Average Scores ===
              Dimension  Average_Score
0    No-Omissions_score       6.866667
1    No-Additions_score       7.000000
2  Grammaticality_score       7.000000
3         Fluency_score       7.000000
